In [1]:
# command to reload libraries
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import os
from pathlib import Path
from sklearn.base import clone


pd.set_option('display.max_columns', None)

In [3]:
REPO_ROOT = Path.cwd().resolve().parent
DATA_PATH = REPO_ROOT / "data"

def load_dataset(file_name):
    df = pd.read_csv(DATA_PATH / file_name)
    return df

def have_more_than_300_samples(dataset):
    return dataset.shape[0] > 300
    


In [4]:
all_datasets_dict = {
    file.strip(".csv"): load_dataset(file) \
    for file in os.listdir(DATA_PATH) if (DATA_PATH / file).is_file()
}

datasets_names = list(all_datasets_dict.keys())

In [5]:
from desReg.des.DESRegression import DESRegression
from desReg.utils import measures as em
from sklearn import linear_model
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

regressors_list = [
    linear_model.Lasso(), 
    linear_model.Ridge(), 
    KNeighborsRegressor(), 
    SVR(), 
    DecisionTreeRegressor()
]

des_ensemble_ih = DESRegression(
    regressors_list = regressors_list,
    random_state=0,
    n_estimators_bag=20,
    competence_region='knn',
    k=5,
    competence_level=em.sum_absolute_error_weighted,
    include_instance_hardness=True,
    include_hardness_measures=False,
    hardness_measures_list=None,
    meta_feature='instance_hardness',
    ensemble_type='DES'
)

drs_ensemble_ih = DESRegression(
    regressors_list = regressors_list,
    random_state=0,
    n_estimators_bag=20,
    competence_region='knn',
    k=5,
    competence_level=em.sum_absolute_error_weighted,
    include_instance_hardness=True,
    include_hardness_measures=False,
    hardness_measures_list=None,
    meta_feature='instance_hardness',
    ensemble_type='DRS'
)

des_all_errors_weighted = DESRegression(
    regressors_list = regressors_list,
    random_state=0,
    n_estimators_bag=20,
    competence_region='knn',
    k=5,
    competence_level=em.all_errors_weighted,
    include_instance_hardness=True,
    include_hardness_measures=False,
    hardness_measures_list=None,
    meta_feature='instance_hardness',
    ensemble_type='DES'
)

des_inverse_ih_all_errors_weighted = DESRegression(
    regressors_list = regressors_list,
    random_state=0,
    n_estimators_bag=20,
    competence_region='knn',
    k=5,
    competence_level=em.all_errors_weighted,
    include_instance_hardness=True,
    include_hardness_measures=False,
    hardness_measures_list=None,
    meta_feature='inverse_instance_hardness',
    ensemble_type='DES'
)

models_templates = {
    "des_ensemble_ih": {"model": des_ensemble_ih},
    'drs_ensemble_ih': {"model": drs_ensemble_ih},
    'des_all_errors_weighted': {"model": des_all_errors_weighted},
    'des_inverse_ih_all_errors_weighted': {"model": des_inverse_ih_all_errors_weighted},
}


/Users/victor.nunes/Documents/Github/regression-instance-hardness/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
# The function, train the model and return the test fold scored and with meta data
dataset_name = 'ocomo_numeri'
model = clone(models_templates['des_ensemble_ih']['model'])
X = all_datasets_dict[dataset_name].drop(columns=['target'])
y = all_datasets_dict[dataset_name]['target']
model.fit(X, y)
y_pred, meta_data = model.predict(X, return_ensemble_sizes=True)

desReg.des.DESRegression - INFO - Calculating instance hardness
desReg.des.DESRegression - INFO - Predicting DES


In [23]:
y_pred_and_metadata = pd.DataFrame(meta_data).fillna(0).assign(y_pred=y_pred)

In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, random_state=42, shuffle=True)

In [ ]:
# Iterate over all datasets
def train_model_over_folds()
X = all_datasets_dict[dataset_name].drop(columns=['target'])
y = all_datasets_dict[dataset_name]['target']

datasets_folds = dict()
datasets_folds[dataset_name] = dict()
dataset_metrics = pd.DataFrame()
for fold_number, (train_index, test_index) in enumerate(kf.split(X)):
    datasets_folds[dataset_name][fold_number+1] = dict(
        X_train = X.iloc[train_index],
        y_train = y.iloc[train_index],
        X_test = X.iloc[test_index],
        y_test = y.iloc[test_index]
    )

    # Train model for each fold
    model = clone(models_templates['des_ensemble_ih']['model'])
    model.fit(
        datasets_folds[dataset_name][fold_number+1]['X_train'], 
        datasets_folds[dataset_name][fold_number+1]['y_train']
    )
    y_pred, meta_data = model.predict(
        datasets_folds[dataset_name][fold_number+1]['X_test'], 
        return_ensemble_sizes=True
    )
    y_pred_and_metadata = pd.DataFrame(meta_data)\
        .fillna(0)\
        .assign(y_pred=y_pred)\
        .assign(y_true=datasets_folds[dataset_name][fold_number+1]['y_test'].values)\
        .assign(fold_number=fold_number+1)

    datasets_folds[dataset_name][fold_number+1]['y_pred'] = y_pred
    datasets_folds[dataset_name][fold_number+1]['y_pred_and_metadata'] = y_pred_and_metadata
    dataset_metrics = pd.concat([dataset_metrics, y_pred_and_metadata], ignore_index=True)

datasets_folds[dataset_name]['all_folds_metrics'] = dataset_metrics

desReg.des.DESRegression - INFO - Calculating instance hardness
desReg.des.DESRegression - INFO - Predicting DES
desReg.des.DESRegression - INFO - Calculating instance hardness
desReg.des.DESRegression - INFO - Predicting DES
desReg.des.DESRegression - INFO - Calculating instance hardness
desReg.des.DESRegression - INFO - Predicting DES
desReg.des.DESRegression - INFO - Calculating instance hardness
desReg.des.DESRegression - INFO - Predicting DES
desReg.des.DESRegression - INFO - Calculating instance hardness
desReg.des.DESRegression - INFO - Predicting DES


In [49]:
import pickle

pickle.dump(datasets_folds, open('../artifacts/datasets_folds.pkl', 'wb'))

/var/folders/4y/1s_blnq50rq_v67_438xyd2m0000gp/T/ipykernel_71800/3796382855.py:3: ResourceWarning: unclosed file <_io.BufferedWriter name='../artifacts/datasets_folds.pkl'>
  pickle.dump(datasets_folds, open('../artifacts/datasets_folds.pkl', 'wb'))


In [51]:
df_analysis = datasets_folds[dataset_name]['all_folds_metrics'].copy()
df_analysis.head()

,Lasso,Ridge,KNeighborsRegressor,DecisionTreeRegressor,TotalRegressors,DistinctRegressors,SVR,y_pred,y_true,fold_number
0,18,20,20,19,77,4,0.0,278.516948,278.0,1
1,17,17,1,19,54,4,0.0,90.937881,60.0,1
2,16,13,14,17,65,5,5.0,69.677704,60.0,1
3,18,16,20,19,73,4,0.0,38.637320,42.0,1
4,19,20,14,17,70,4,0.0,829.952803,324.0,1


In [ ]:
datasets_regression_metrics = pd.DataFrame()

In [52]:
# Calculate mse of all folds
df_analysis['mse'] = (df_analysis['y_true'] - df_analysis['y_pred']) ** 2
df_analysis.groupby('fold_number')['mse'].mean()

fold_number
1    214008.619028
2      2190.581115
3    132957.485114
4    125445.202184
5      5287.900594
Name: mse, dtype: float64

In [53]:
df_analysis['mse'].mean()


95977.95760701214

In [ ]:
datasets_folds = dict()
for name, dataset in all_datasets_dict.items():
    X = dataset.drop(columns=['target'])
    y = dataset['target']
    datasets_folds[name] = dict()
    for fold_number, (train_index, test_index) in enumerate(kf.split(X)):
        datasets_folds[name][fold_number+1] = dict(
            X_train = X.iloc[train_index],
            y_train = y.iloc[train_index],
            X_test = X.iloc[test_index],
            y_test = y.iloc[test_index]
        )
